## Notebook for the identification of the rsids for a given list of variants
1. Get list of variants based on chrom-pos-ref-alt
2. Identify the associated rsid 
3.  

In [116]:
import pandas as pd
import math
import yaml

# config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf



In [117]:
# read the variants:
variant_list_path = config['files']['collaborations']['langenberg_significant_vaiants_unique']
variant_list_df = pd.read_csv(variant_list_path, sep="\t")
variant_list_df

,SPDI,variant_logFC,variant_adj_P_Val
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132
1,NC_000005.10:14259915:C:T,1.119690,1.515352e-72
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61
...,...,...,...
809,NC_000003.12:71497206:G:A,-0.128911,4.365701e-02
810,NC_000004.12:5758816:C:T,-0.128509,4.649902e-02
811,NC_000020.11:8366413:G:A,0.130123,4.878092e-02
812,NC_000010.11:113007099:T:A,0.133877,4.933347e-02


In [118]:
# get chrom and postion of table
def extract_chromosome_number(spdi):
    """Extract chromosome number or 'X' from SPDI identifier."""
    try:
        # Mapping of chromosome names to numbers
        chrom_map = {
            "NC_000001.11": "1", "NC_000002.12": "2", "NC_000003.12": "3", "NC_000004.12": "4",
            "NC_000005.10": "5", "NC_000006.12": "6", "NC_000007.14": "7", "NC_000008.11": "8",
            "NC_000009.12": "9", "NC_000010.11": "10", "NC_000011.10": "11", "NC_000012.12": "12",
            "NC_000013.11": "13", "NC_000014.9": "14", "NC_000015.10": "15", "NC_000016.10": "16",
            "NC_000017.11": "17", "NC_000018.10": "18", "NC_000019.10": "19", "NC_000020.11": "20",
            "NC_000021.9": "21", "NC_000022.11": "22", "NC_000023.11": "X", "NC_000024.10": "Y"
        }

        # Extract chromosome name from SPDI (before the first ':')
        chrom_name = spdi.split(':')[0]

        # Get corresponding chromosome number or "X" from the map
        chrom_number = chrom_map.get(chrom_name, chrom_name)  # Default to chrom_name if not found

        return chrom_number
    except Exception as e:
        return f"Error parsing chromosome from SPDI: {str(e)}"

def extract_position(spdi):
    try:
        # Split the SPDI by ':'
        parts = spdi.split(':')

        # Extract position
        position = int(parts[1])  # Genomic position (e.g., 396320)

        return position
    except Exception as e:
        return f"Error parsing position from SPDI: {str(e)}"

def extract_ref(spdi):
    try:
        # Split the SPDI by ':'
        parts = spdi.split(':')

        # Extract reference
        reference = parts[2]  # Genomic reference (C)

        return reference
    except Exception as e:
        return f"Error parsing reference from SPDI: {str(e)}"

def extract_alt(spdi):
    try:
        # Split the SPDI by ':'
        parts = spdi.split(':')

        # Extract alternative
        alternative = parts[3]  # Genomic alternative (e.g., T)

        return alternative
    except Exception as e:
        return f"Error parsing alternative from SPDI: {str(e)}"

variant_list_df['chrom'] = variant_list_df['SPDI'].apply(extract_chromosome_number)
variant_list_df['pos'] = variant_list_df['SPDI'].apply(extract_position)
variant_list_df['ref'] = variant_list_df['SPDI'].apply(extract_ref)
variant_list_df['alt'] = variant_list_df['SPDI'].apply(extract_alt)

In [ ]:
# get rsid from position
# import myvariant

# # Initialize the MyVariant client
# mv = myvariant.MyVariant()

# # List of variants in chr-pos-ref-alt format
# variants = [
#     "6-396321-C-T",
#     "1-866422-C-T",
#     "1-876664-G-A",
#     "1-69635-G-C"
# ]

# # Convert variants to HGVS format
# hgvs_variants = [f"chr{var.split('-')[0]}:g.{var.split('-')[1]}{var.split('-')[2]}>{var.split('-')[3]}" for var in variants]

# # Fetch variant information
# results = mv.getvariants(hgvs_variants, fields="dbsnp.rsid")

# # Process and print results
# for variant, result in zip(variants, results):
#     rsid = result.get('dbsnp', {}).get('rsid', 'N/A')
#     print(f"Variant: {variant}, rsID: {rsid}")

# import requests

# def get_rsid_from_position(chromosome, position, species="human"):
#     """
#     Query Ensembl to retrieve the RSID for a given genomic position.

#     Args:
#         chromosome (str): Chromosome number (e.g., "1", "X").
#         position (int): Genomic position on the chromosome.
#         species (str): Species name (default: "human").

#     Returns:
#         str: RSID if found, or a message indicating no RSID.
#     """
#     # Construct the URL
#     url = f"https://rest.ensembl.org/overlap/region/{species}/{chromosome}:{position}-{position}?"

#     # Headers to specify JSON response
#     headers = {"Content-Type": "application/json"}

#     # Make the request
#     response = requests.get(url, headers=headers)

#     # Handle the response
#     if response.status_code == 200:
#         data = response.json()
#         # Look for RSID in the response
#         for item in data:
#             if item.get("id", "").startswith("rs"):
#                 return item["id"]
#         return "No RSID found for the given position."
#     else:
#         print(response)
#         return f"Error: Unable to fetch data (status code: {response.status_code})"

import requests

def get_rsid_from_position(chromosome, position, reference, alternative, species="homo_sapiens"):
    """
    Query the Ensembl REST API to retrieve the RSID for a given genomic position.

    Args:
        chromosome (str): Chromosome number (e.g., "1", "X").
        position (int): Genomic position on the chromosome.
        species (str): Species name (default: "homo_sapiens").

    Returns:
        str: RSID if found, or a message indicating no RSID.
    """
    # Construct the region string
    region = f"{chromosome}:{position}-{position}"

    # API endpoint URL
    url = f"https://rest.ensembl.org/overlap/region/{species}/{region}"

    # Query parameters
    params = {
        "feature": "variation"  # Limit to variation features (includes RSIDs)
    }

    # Headers to specify JSON response
    headers = {"Content-Type": "application/json"}

    try:
        # Make the request
        response = requests.get(url, headers=headers, params=params)

        # Raise an error for bad HTTP responses
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()

        # Extract RSIDs based on given ref and alt alleles
        rsids = [item["id"] for item in data if item.get("id", "").startswith("rs") and item.get('alleles', "")[0] == reference and item.get('alleles', '')[1] == alternative]
        if len(rsids) > 1:
            raise ValueError("Mutliple rsids for one SNV are not possible...")
        if rsids:
            return rsids[0]
            return f"RSID(s) found: {', '.join(rsids)}"
        else:
            return None
            return "No RSID found for the given position."
    except requests.exceptions.RequestException as e:
        return f"Error querying Ensembl API: {e}"

variant_list_df['rsid'] = variant_list_df.apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
variant_list_df



# only on subset for testing:
# variant_list_df_head = variant_list_df.head(n=7)
# variant_list_df_head['rsid'] = variant_list_df_head.apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
# variant_list_df_head['n_rsid'] = variant_list_df_head['rsid'].apply(len)
# variant_list_df_head['chrom'].value_counts()
# variant_list_df_head
# variant_list_df_head = variant_list_df.head(n=100)
# variant_list_df_head['rsid'] = variant_list_df_head.apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
# variant_list_df_head
# variant_list_df_head.loc[variant_list_df_head['n_rsid'] >1]
# variant_list_df_head.loc[variant_list_df_head['n_rsid'] >1].apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
# # Example usage 6-396321-C-T
# chromosome = "6"
# position = 396321
# rsid = get_rsid_from_position(chromosome, position)
# print(f"RSID for {chromosome}:{position} is {rsid}")

# expected: rs2046495165 rs1740377627 (for the rows with multiple rsids)

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos,ref,alt,rsid
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132,5,14408058,A,G,[]
1,NC_000005.10:14259915:C:T,1.119690,1.515352e-72,5,14259915,C,T,[]
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69,1,231734632,A,G,[rs543065973]
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64,2,219267000,T,C,[]
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61,3,142578094,C,G,[]
...,...,...,...,...,...,...,...,...
809,NC_000003.12:71497206:G:A,-0.128911,4.365701e-02,3,71497206,G,A,[]
810,NC_000004.12:5758816:C:T,-0.128509,4.649902e-02,4,5758816,C,T,[]
811,NC_000020.11:8366413:G:A,0.130123,4.878092e-02,20,8366413,G,A,[]
812,NC_000010.11:113007099:T:A,0.133877,4.933347e-02,10,113007099,T,A,[]


In [110]:
variant_list_df['chrom'].value_counts()

chrom
1     87
6     65
3     53
14    51
7     49
11    47
16    44
2     39
15    36
10    34
9     31
18    30
5     30
17    29
8     28
12    27
4     27
22    26
X     20
19    20
13    16
20    14
21    11
Name: count, dtype: int64

In [ ]:
variant_list_df

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos,ref,alt,rsid,n_rsid
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132,5,14408058,A,G,[],0
1,NC_000005.10:14259915:C:T,1.119690,1.515352e-72,5,14259915,C,T,[],0
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69,1,231734632,A,G,[rs543065973],1
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64,2,219267000,T,C,[],0
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61,3,142578094,C,G,[],0
5,NC_000011.10:35289965:C:T,0.833250,1.213270e-51,11,35289965,C,T,[],0
6,NC_000016.10:68784389:A:G,0.728294,3.003789e-47,16,68784389,A,G,[],0


In [113]:
print('Number of significant variants with rsid: ', variant_list_df.loc[variant_list_df['n_rsid'] >0].shape[0])
variant_list_df['n_rsid'] = variant_list_df['rsid'].apply(len)
variant_list_df.loc[variant_list_df['n_rsid'] >1] # []
variant_list_df.loc[variant_list_df['n_rsid'] >0] # []


Number of significant variants with rsid:  38


,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos,ref,alt,rsid,n_rsid
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69,1,231734632,A,G,[rs543065973],1
22,NC_000008.11:22998251:A:G,-0.616552,1.377095e-22,8,22998251,A,G,[rs1241820132],1
30,NC_000001.11:162145084:A:G,-0.971605,1.661929e-19,1,162145084,A,G,[rs1048660947],1
33,NC_000020.11:32130529:A:G,-0.868284,7.347568e-18,20,32130529,A,G,[rs2046495165],1
64,NC_000006.12:360501:G:A,-0.638457,3.904407e-13,6,360501,G,A,[rs1447623934],1
84,NC_000006.12:112155944:G:A,0.514565,2.773851e-11,6,112155944,G,A,[rs1314015278],1
94,NC_000004.12:119106333:G:A,0.316681,2.498489e-10,4,119106333,G,A,[rs1740377627],1
128,NC_000014.9:65081458:G:A,-0.315078,1.404551e-07,14,65081458,G,A,[rs563197987],1
130,NC_000014.9:23437065:G:A,-0.545211,1.511800e-07,14,23437065,G,A,[rs1893184564],1
187,NC_000003.12:125214055:C:T,0.369430,5.315528e-06,3,125214055,C,T,[rs2125189856],1


In [102]:
# number of variants with rsids
# variant_list_df.loc[variant_list_df['n_rsid'] >0].shape[0] # 38
variant_list_df_head.loc[variant_list_df['n_rsid'] >0].shape[0] # 38

KeyError: 'n_rsid'

In [114]:
variant_list_df.loc[variant_list_df['n_rsid'] >0]

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos,ref,alt,rsid,n_rsid
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69,1,231734632,A,G,[rs543065973],1
22,NC_000008.11:22998251:A:G,-0.616552,1.377095e-22,8,22998251,A,G,[rs1241820132],1
30,NC_000001.11:162145084:A:G,-0.971605,1.661929e-19,1,162145084,A,G,[rs1048660947],1
33,NC_000020.11:32130529:A:G,-0.868284,7.347568e-18,20,32130529,A,G,[rs2046495165],1
64,NC_000006.12:360501:G:A,-0.638457,3.904407e-13,6,360501,G,A,[rs1447623934],1
84,NC_000006.12:112155944:G:A,0.514565,2.773851e-11,6,112155944,G,A,[rs1314015278],1
94,NC_000004.12:119106333:G:A,0.316681,2.498489e-10,4,119106333,G,A,[rs1740377627],1
128,NC_000014.9:65081458:G:A,-0.315078,1.404551e-07,14,65081458,G,A,[rs563197987],1
130,NC_000014.9:23437065:G:A,-0.545211,1.511800e-07,14,23437065,G,A,[rs1893184564],1
187,NC_000003.12:125214055:C:T,0.369430,5.315528e-06,3,125214055,C,T,[rs2125189856],1


In [115]:
variant_list_df.to_csv('20241119_significant_variants_with_rsid.tsv', sep="\t", index=False)

In [88]:
import os
from Bio import Entrez
# from pydantic import BaseModel, Field
# from openai import OpenAI


# # Required: Export your API Key (on linux: export OPENAI_API_KEY="..." )
# client = OpenAI(
#   api_key=os.environ['OPENAI_API_KEY']
# )

# class Format(BaseModel):
#     rs_id: str = Field(..., description="rsid to check in the abstract")
#     rs_mentioned: bool = Field(..., description="True if the rsid is mentioned in the given abstract, otherwise false")
#     condition: str = Field(..., description="Associated medical condition or phenotype")
#     effect: str = Field(..., description="Effect of the rsid on the condition mentioned in the abstract")
#     celltype: str = Field(..., description="Most likely cell type the rsid is associated with")

# def check_abstract(rs_id_to_check, abstract):
#     prompt = f"""
#     You are provided with the following abstract:
#     ---
#     {abstract}
#     ---
#     Determine the following:
#     1. rs_id: "{rs_id_to_check}"
#     2. rs_id_mentioned: Does the given rsid "{rs_id_to_check}" appear in the abstract? Set it to True if found, otherwise False.
#     3. condition: The associated medical condition or phenotype mentioned in the abstract related to the rsid. Empty string if not mentioned.
#     4. effect: "The effect of the rsid "{rs_id_to_check}" on the condition mentioned in the abstract. Empty string if not mentioned"
#     5. celltype: "The cell type which the rsid and condition is associated with according to the abstract. Empty string if not mentioned."
#     """

#     # Query the model
#     response = client.beta.chat.completions.parse(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant."},
#             {"role": "user", "content": prompt},
#         ],
#         response_format=Format,
#         #temperature=0,  # Deterministic output
#         #max_tokens=150,  # Adjust based on response size needed
#     )

#     output = response.choices[0].message.parsed
#     return output

def get_rsid_list(variant_list_df):
    # ...
    # return ["rs12203592"]
    variant_rsids = variant_list_df.loc[variant_list_df['n_rsid'] > 0]
    variant_rsids['rsid_unlisted'] = variant_rsids['rsid'].apply(lambda rsid: rsid[0])
    return variant_rsids['rsid_unlisted'].to_list()

# Set your email (required by NCBI policy)
Entrez.email = "your_email@example.com"

# Function to fetch PubMed articles for an rsID
def fetch_pubmed_articles(rsid):
    query = f"{rsid}[All Fields]"
    handle = Entrez.esearch(db="pubmed", term=query, retmax=50)  # Adjust retmax as needed
    record = Entrez.read(handle)
    handle.close()
    print(record)
    # Get list of PubMed IDs (PMIDs)
    pmids = record["IdList"]
    return pmids

# Function to fetch article abstract
def fetch_abstract(pmid):
    handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="text")
    abstract = handle.read()
    handle.close()
    return abstract

rsid_list = get_rsid_list(variant_list_df)

D = {}

for rsid in rsid_list:
    if rsid not in D.keys():
        D[rsid] = {}
    pmid_list = fetch_pubmed_articles(rsid)
    D[rsid] = pmid_list
    # for pmid in pmid_list:
    #     print(f"RSID: {rsid}, PubMed ID: {pmid}")
    #     abstract = fetch_abstract(pmid)
        # output = check_abstract(rsid, abstract)
        # print([output.condition, output.rs_mentioned, output.effect, output.celltype])
        # D[rsid][pmid] = [output.condition, output.rs_mentioned, output.effect, output.celltype]

print("OUTPUT:")
print(D)

/tmp/ipykernel_185966/1985092740.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  variant_rsids['rsid_unlisted'] = variant_rsids['rsid'].apply(lambda rsid: rsid[0])


{'Count': '0', 'RetMax': '0', 'RetStart': '0', 'IdList': [], 'TranslationSet': [], 'QueryTranslation': 'rs543065973[All Fields]', 'ErrorList': {'PhraseNotFound': ['rs543065973'], 'FieldNotFound': []}, 'WarningList': {'QuotedPhraseNotFound': [], 'OutputMessage': ['No items found.'], 'PhraseIgnored': []}}
{'Count': '0', 'RetMax': '0', 'RetStart': '0', 'IdList': [], 'TranslationSet': [], 'QueryTranslation': 'rs1241820132[All Fields]', 'ErrorList': {'PhraseNotFound': ['rs1241820132'], 'FieldNotFound': []}, 'WarningList': {'QuotedPhraseNotFound': [], 'OutputMessage': ['No items found.'], 'PhraseIgnored': []}}
{'Count': '0', 'RetMax': '0', 'RetStart': '0', 'IdList': [], 'TranslationSet': [], 'QueryTranslation': 'rs1048660947[All Fields]', 'ErrorList': {'PhraseNotFound': ['rs1048660947'], 'FieldNotFound': []}, 'WarningList': {'QuotedPhraseNotFound': [], 'OutputMessage': ['No items found.'], 'PhraseIgnored': []}}
{'Count': '0', 'RetMax': '0', 'RetStart': '0', 'IdList': [], 'TranslationSet': []

In [85]:
fetch_pubmed_articles('rs543065973')

[]

In [37]:
variant_list_df_head['rsid'].isna().sum()

58

In [1]:
from Bio import Entrez

# Set your email (required by NCBI policy)
Entrez.email = "your_email@example.com"

# Function to fetch PubMed articles for an rsID
def fetch_pubmed_articles(rsid):
    query = f"{rsid}[All Fields]"
    handle = Entrez.esearch(db="pubmed", term=query, retmax=10)  # Adjust retmax as needed
    record = Entrez.read(handle)
    handle.close()

    # Get list of PubMed IDs (PMIDs)
    pmids = record["IdList"]
    return pmids

# Example usage
rsid_list = ["rs12203592"]
for rsid in rsid_list:
    pmids = fetch_pubmed_articles(rsid)
    print(f"RSID: {rsid}, PubMed IDs: {pmids}")

RSID: rs12203592, PubMed IDs: ['39075179', '37902747', '35390444', '34898573', '34424336', '34418235', '34293285', '33342058', '32856602', '32121219']


In [ ]:
from Bio import Entrez

# Set your email (required by NCBI policy)
Entrez.email = "your_email@example.com"

# Function to fetch article details
def fetch_article_details(pmids):
    handle = Entrez.efetch(db="pubmed", id=",".join(pmids), rettype="abstract", retmode="text")
    details = handle.read()
    handle.close()
    return details

# Example usage
pmid_list = ["39075179"]  # Replace with actual PubMed IDs
article_details = fetch_article_details(pmid_list)
print(article_details)